In [1]:
import argparse
import os
import joblib
import pandas as pd
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from palmerpenguins import load_penguins
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


# Clase DataProcessor

Esta clase se encarga de todo el flujo de procesamiento de datos:

1. Cargar el dataset.
2. Limpiar datos faltantes o duplicados.
3. Transformar variables categóricas.
4. Extraer las variables independientes (X) y la variable objetivo (y).

Centraliza toda la lógica de preparación de datos para mantener el código modular y reutilizable.

In [2]:
class DataProcessor:
    """Handles all data loading, cleaning, transformation, and feature extraction operations."""
    
    def __init__(self):
        """Initialize DataProcessor.
        """
        self.df = None
        self.X = None
        self.y = None
    
    def load_data(self):
        """Load data using palmerpenguins load_penguins function."""
        self.df = load_penguins()
        return self
    
    def clean_data(self):
        """Remove missing values and duplicate rows."""
        self.df = self.df.dropna()
        self.df = self.df.drop_duplicates()
        return self
    
    def transform_data(self):
        """Transform species column to numeric and create dummy variables."""
        # Transform species column where Adelie=0, Chinstrap=1, Gentoo=2
        species_mapping = {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}
        self.df['species'] = self.df['species'].map(species_mapping)
        
        # Dummy encode categorical columns
        self.df = pd.get_dummies(self.df)
        return self
    
    def extract_features_target(self, target_column='species'):
        """Extract features and target variable from dataframe.
        
        Args:
            target_column (str): Name of the target column.
        """
        self.X = self.df.drop(columns=[target_column])
        self.y = self.df[target_column]
        return self
    
    def process(self, target_column='species'):
        """Run the complete data processing pipeline.
        
        Args:
            target_column (str): Name of the target column.
            
        Returns:
            tuple: X (features) and y (target) dataframes.
        """
        self.load_data()
        self.clean_data()
        self.transform_data()
        self.extract_features_target(target_column)
        return self.X, self.y

# División del Dataset

Esta función divide el dataset en tres conjuntos:

- **Train**: para entrenar el modelo.
- **Validation**: para ajustar hiperparámetros y evaluar durante entrenamiento.
- **Test**: para evaluación final del modelo.

Se usa un `random_state` para garantizar reproducibilidad.

In [3]:
def split_data(X, y, test_size=0.3, val_size=0.5, random_state=42):
    """Split data into train, validation, and test sets.
    
    Args:
        X: Features dataframe.
        y: Target variable.
        test_size (float): Proportion of data for testing.
        val_size (float): Proportion of temp data for validation.
        random_state (int): Random seed for reproducibility.
        
    Returns:
        tuple: (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=val_size, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# Clase Model

Esta clase encapsula toda la lógica relacionada con el modelo:

- Construcción del modelo según el tipo seleccionado.
- Entrenamiento.
- Validación.
- Evaluación en test.
- Exportación a archivo `.pkl`.

Permite cambiar fácilmente entre distintos algoritmos sin modificar el resto del pipeline.

In [4]:
class Model:
    """Handles model building, training, validation, and export operations."""
    
    def __init__(self, model_type='svm', **model_params):
        """Initialize Model with model type and parameters.
        
        Args:
            model_type (str): Type of model to build ('svm', 'logistic_regression', 'random_forest').
            **model_params: Additional parameters for the model.
        """
        self.model_type = model_type
        self.model_params = model_params
        self.model = None
        self.X_train = None
        self.X_val = None
        self.X_test = None
        self.y_train = None
        self.y_val = None
        self.y_test = None
    
    def set_data(self, X_train, X_val, X_test, y_train, y_val, y_test):
        """Set training, validation, and test data.
        
        Args:
            X_train: Training features.
            X_val: Validation features.
            X_test: Test features.
            y_train: Training target.
            y_val: Validation target.
            y_test: Test target.
        """
        self.X_train = X_train
        self.X_val = X_val
        self.X_test = X_test
        self.y_train = y_train
        self.y_val = y_val
        self.y_test = y_test
        return self
    
    def build_model(self):
        """Build model based on model type and parameters."""
        if self.model_type == 'svm':
            self.model = SVC(**self.model_params)
        elif self.model_type == 'logistic_regression':
            self.model = LogisticRegression(**self.model_params)
        elif self.model_type == 'random_forest':
            self.model = RandomForestClassifier(**self.model_params)
        else:
            raise ValueError(f'Unsupported model type: {self.model_type}')
        return self
    
    def train(self):
        """Train the model on training data."""
        if self.model is None:
            raise ValueError('Model not built. Call build_model() first.')
        self.model.fit(self.X_train, self.y_train)
        return self
    
    def validate(self):
        """Validate the model on validation data.
        
        Returns:
            str: Classification report.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        predictions = self.model.predict(self.X_val)
        report = classification_report(self.y_val, predictions)
        return report
    
    def test(self):
        """Test the model on test data.
        
        Returns:
            str: Classification report.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        predictions = self.model.predict(self.X_test)
        report = classification_report(self.y_test, predictions)
        return report
    
    def export(self, file_path):
        """Export trained model to file.
        
        Args:
            file_path (str): Path to save the model file.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        
        # Validate if folder exists, if not create it
        folder = os.path.dirname(file_path)
        if not os.path.exists(folder):
            os.makedirs(folder)
        
        joblib.dump(self.model, file_path)
        return self

# Configuración del Entrenamiento

En este bloque se:

- Define la carpeta donde se guardarán los modelos.
- Se crea la carpeta si no existe.
- Se selecciona el tipo de modelo a entrenar.
- Se define el nombre con el que se guardará el modelo entrenado.

Esto permite controlar dinámicamente qué modelo se entrena y cómo se guarda.

In [15]:
models_folder = "/models"
os.makedirs(models_folder, exist_ok=True)

model_type = "svm"  
model_save_name = "svm_v2"

# Preprocesamiento de Datos

Aquí se ejecuta el pipeline de procesamiento:

1. Se instancia `DataProcessor`.
2. Se generan las variables X (features) y y (target).
3. Se divide el dataset en train, validation y test.
4. Se imprimen los tamaños de cada conjunto.

Este bloque solo se ejecuta una vez por entrenamiento.

In [16]:
print(f'{"="*60}')
print('DATA PREPROCESSING')
print(f'{"="*60}')

data_processor = DataProcessor()
X, y = data_processor.process(target_column='species')

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print(f'Train samples: {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Test samples: {len(X_test)}')

DATA PREPROCESSING
Train samples: 233
Validation samples: 50
Test samples: 50


# Configuración de Modelos Disponibles

Se define un diccionario con los modelos disponibles y sus hiperparámetros.

Luego se valida que el `model_type` seleccionado exista dentro de las opciones configuradas.

Esto permite flexibilidad para experimentar con distintos algoritmos.

In [17]:
model_configs = {
    'svm': {'kernel': 'rbf', 'C': 1.0},
    'logistic_regression': {'max_iter': 1000, 'random_state': 42},
    'random_forest': {'n_estimators': 100, 'random_state': 42}
}

if model_type not in model_configs:
    raise ValueError(f"Model type '{model_type}' is not supported")

model_params = model_configs[model_type]

# Entrenamiento y Validación

En este bloque:

1. Se instancia el modelo seleccionado.
2. Se cargan los datos de entrenamiento, validación y test.
3. Se construye el modelo.
4. Se entrena.
5. Se valida y se imprimen métricas.

Aquí ocurre el aprendizaje del modelo.

In [18]:
print(f'\n{"="*60}')
print(f'TRAINING {model_type.upper()} MODEL')
print(f'{"="*60}')

model = Model(model_type, **model_params)
model.set_data(X_train, X_val, X_test, y_train, y_val, y_test)

print('Building model...')
model.build_model()

print('Training model...')
model.train()

print('Validating model...')
validation_report = model.validate()
print(validation_report)


TRAINING SVM MODEL
Building model...
Training model...
Validating model...
              precision    recall  f1-score   support

           0       0.67      0.91      0.77        22
           1       0.00      0.00      0.00        10
           2       0.90      1.00      0.95        18

    accuracy                           0.76        50
   macro avg       0.52      0.64      0.57        50
weighted avg       0.62      0.76      0.68        50



/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [19]:
print(f'\n{"="*60}')
print('TEST RESULTS')
print(f'{"="*60}')

test_report = model.test()
print(test_report)

print(f'\n{"="*60}')
print('MODEL TRAINED, VALIDATED AND TESTED SUCCESSFULLY')
print(f'Saved in: {model_save_name}')
print(f'{"="*60}')


TEST RESULTS
              precision    recall  f1-score   support

           0       0.62      0.88      0.73        26
           1       0.00      0.00      0.00        13
           2       0.77      0.91      0.83        11

    accuracy                           0.66        50
   macro avg       0.46      0.60      0.52        50
weighted avg       0.49      0.66      0.56        50


MODEL TRAINED, VALIDATED AND TESTED SUCCESSFULLY
Saved in: svm_v2


/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Exportación del Modelo

En este bloque:

1. Se construye la ruta del archivo usando el nombre definido.
2. Se guarda el modelo entrenado en formato `.pkl`.
3. Se confirma que el guardado fue exitoso.

Este archivo será luego usado por la API para realizar predicciones.

In [20]:
model_file = os.path.join(models_folder, f'{model_save_name}.pkl')

print(f'\nSaving model as: {model_file}')
model.export(model_file)

print('Model saved successfully.')


Saving model as: /models/svm_v2.pkl
Model saved successfully.
